# Columna vertical bajo su propio peso: solución por diferencias finitas como problema de contorno

Este notebook presenta un ejemplo didáctico para un curso de **Métodos Numéricos**. Se estudia una **columna vertical en voladizo** (empotrada en la base y libre en el extremo superior) que empieza a doblarse por efecto de su **propio peso**.

La idea central aquí es **no** formular el problema directamente como uno de autovalores, porque eso puede confundir a estudiantes que todavía están aprendiendo diferencias finitas. En su lugar, se plantea un **problema de contorno** con una condición de escala simple.

## 1. Modelo diferencial

Sea $x$ la coordenada medida desde la base empotrada hasta el extremo libre, con $0 \le x \le L$. Si la columna tiene peso por unidad de longitud $w = \rho A g$, entonces la fuerza axial interna en una sección situada en $x$ es

$P(x) = w(L-x)$

porque esa sección soporta el peso del tramo que queda por encima.

Si $\theta(x)$ es el ángulo que forma la tangente de la columna con la vertical, la ecuación no lineal de equilibrio puede escribirse como

$EI \, \dfrac{d^2\theta}{dx^2} + P(x)\sin\theta = 0$

es decir,

$EI \, \dfrac{d^2\theta}{dx^2} + w(L-x)\sin\theta = 0$

donde:

- $E$ es el módulo de Young,
- $I$ es el segundo momento de área,
- $\rho$ es la densidad del material,
- $A$ es el área de la sección transversal,
- $g$ es la aceleración gravitacional.

## 2. Forma adimensional

Definimos la coordenada adimensional

$\xi = \dfrac{x}{L}$

entonces

$\dfrac{d}{dx} = \dfrac{1}{L}\dfrac{d}{d\xi}, \qquad \dfrac{d^2}{dx^2} = \dfrac{1}{L^2}\dfrac{d^2}{d\xi^2}$

y la ecuación queda

$\dfrac{d^2\theta}{d\xi^2} + \lambda (1-\xi)\sin\theta = 0$

donde

$\lambda = \dfrac{wL^3}{EI}$

es el parámetro adimensional que compara el efecto del peso propio con la rigidez flexional.

## 3. Aproximación lineal

Cerca del inicio del pandeo, las deflexiones son pequeñas y se usa

$\sin\theta \approx \theta$

de modo que el problema linealizado es

$\dfrac{d^2\theta}{d\xi^2} + \lambda (1-\xi)\theta = 0$

## 4. Condiciones de frontera

Para evitar introducir directamente un problema de autovalores, aquí imponemos las siguientes condiciones:

$\theta(0)=0$

$\dfrac{d\theta}{d\xi}(1)=1$

La primera representa el **empotramiento** en la base.  
La segunda fija una **escala de solución** en el extremo libre. En otras palabras, se fuerza una pendiente unitaria en la punta, de manera que el problema se convierte en un problema de contorno ordinario.

### Idea didáctica

Con esta formulación, para cada valor de $\lambda$ resolvemos un sistema lineal por diferencias finitas. Cuando $\lambda$ se aproxima al valor crítico, la matriz del sistema se vuelve casi singular y la respuesta crece mucho. Esa amplificación es una forma numérica clara de identificar la cercanía al pandeo crítico.

## 5. Comentario sobre la solución analítica

La ecuación linealizada

$\theta'' + \lambda(1-\xi)\theta = 0$

puede transformarse a una ecuación de Airy. Si se define una variable adecuada del tipo

$z = \lambda^{1/3}(1-\xi)$

la ecuación se reescribe en una forma equivalente a

$\dfrac{d^2\theta}{dz^2} - z\,\theta = 0$

cuyas soluciones se expresan en términos de las funciones de Airy $Ai(z)$ y $Bi(z)$.

Por tanto, la teoría exacta del caso linealizado sí involucra funciones especiales. Sin embargo, en este notebook el objetivo principal es mostrar cómo resolver el problema con **diferencias finitas**, de forma clara y pedagógica.

## 6. Discretización por diferencias finitas

Dividimos el intervalo $[0,1]$ en $N$ subintervalos, con paso

$h = \dfrac{1}{N}$

y nodos $\xi_i = ih$, con $i=0,1,\dots,N$.

Para los nodos interiores, usamos la aproximación centrada:

$\dfrac{d^2\theta}{d\xi^2}\bigg|_{\xi_i} \approx \dfrac{\theta_{i-1} - 2\theta_i + \theta_{i+1}}{h^2}$

de modo que la ecuación discreta queda

$\dfrac{\theta_{i-1} - 2\theta_i + \theta_{i+1}}{h^2} + \lambda (1-\xi_i)\theta_i = 0$

o equivalentemente,

$\theta_{i-1} + \left[-2 + h^2\lambda(1-\xi_i)\right]\theta_i + \theta_{i+1} = 0$

La condición en la base es

$\theta_0 = 0$

y para la condición en el extremo libre usamos una diferencia hacia atrás de primer orden:

$\dfrac{d\theta}{d\xi}(1) \approx \dfrac{\theta_N - \theta_{N-1}}{h} = 1$

es decir,

$-\theta_{N-1} + \theta_N = h$

Así se obtiene, para cada valor de $\lambda$, un sistema lineal

$A(\lambda)\,\Theta = b$

que puede resolverse con álgebra lineal estándar.

## 7. Qué hará el código

El código en una sola celda hará lo siguiente:

1. Construirá la matriz de diferencias finitas para un valor dado de $\lambda$.
2. Resolverá el problema de contorno linealizado.
3. Barrerá muchos valores de $\lambda$.
4. Medirá la amplitud de la respuesta, por ejemplo mediante $\max |\theta|$.
5. Mostrará que esa amplitud crece fuertemente cerca del valor crítico.
6. Dibujará formas de la columna para algunos valores de $\lambda$.
7. Estimará el punto crítico como el valor donde la matriz tiene la **menor singularidad** o donde la amplitud se dispara.



In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import solve, svd, LinAlgError

# ============================================================
# 1. PARÁMETROS NUMÉRICOS
# ============================================================
N = 200                           # número de subintervalos
h = 1.0 / N
xi = np.linspace(0.0, 1.0, N + 1)

# Rango de barrido del parámetro adimensional lambda = w L^3 / (EI)
lam_min = 0.1
lam_max = 12.0
n_lams = 500
lam_vals = np.linspace(lam_min, lam_max, n_lams)

# ============================================================
# 2. FUNCIÓN PARA CONSTRUIR Y RESOLVER EL SISTEMA LINEAL
#    theta'' + lambda (1-xi) theta = 0
#    BCs: theta(0)=0, theta'(1)=1
# ============================================================
def resolver_bvp(lam, N):
    h = 1.0 / N
    xi = np.linspace(0.0, 1.0, N + 1)
    A = np.zeros((N + 1, N + 1), dtype=float)
    b = np.zeros(N + 1, dtype=float)

    # BC en xi = 0: theta(0) = 0
    A[0, 0] = 1.0
    b[0] = 0.0

    # Ecuaciones interiores
    for i in range(1, N):
        A[i, i - 1] = 1.0
        A[i, i]     = -2.0 + h**2 * lam * (1.0 - xi[i])
        A[i, i + 1] = 1.0
        b[i] = 0.0

    # BC en xi = 1: theta'(1) = 1
    # (theta_N - theta_{N-1}) / h = 1
    A[N, N - 1] = -1.0
    A[N, N] = 1.0
    b[N] = h

    theta = solve(A, b)

    # Indicadores numéricos
    svals = svd(A, compute_uv=False)
    sigma_min = svals[-1]
    cond_aprox = svals[0] / svals[-1]

    return xi, theta, A, sigma_min, cond_aprox

# ============================================================
# 3. BARRIDO EN lambda
# ============================================================
max_theta = []
theta_tip = []
sigma_min_vals = []
cond_vals = []

for lam in lam_vals:
    try:
        xi_num, theta_num, A_num, sigma_min, cond_aprox = resolver_bvp(lam, N)
        max_theta.append(np.max(np.abs(theta_num)))
        theta_tip.append(theta_num[-1])
        sigma_min_vals.append(sigma_min)
        cond_vals.append(cond_aprox)
    except LinAlgError:
        max_theta.append(np.nan)
        theta_tip.append(np.nan)
        sigma_min_vals.append(np.nan)
        cond_vals.append(np.nan)

max_theta = np.array(max_theta)
theta_tip = np.array(theta_tip)
sigma_min_vals = np.array(sigma_min_vals)
cond_vals = np.array(cond_vals)

# Estimación del lambda crítico:
# donde sigma_min es mínima (matriz casi singular)
idx_crit = np.nanargmin(sigma_min_vals)
lam_crit_est = lam_vals[idx_crit]

print("="*72)
print("COLUMNA BAJO SU PROPIO PESO - FORMULACIÓN COMO BVP")
print("="*72)
print(f"Número de subintervalos N = {N}")
print(f"Estimación numérica de lambda crítico = {lam_crit_est:.6f}")
print("Interpretación: cerca de este valor la matriz A(lambda) se vuelve")
print("casi singular y la respuesta del BVP con theta'(1)=1 se amplifica mucho.")
print("="*72)

# ============================================================
# 4. SOLUCIONES PARA ALGUNOS VALORES DE lambda
# ============================================================
lam_ejemplos = [2.0, 5.0, lam_crit_est - 0.10, lam_crit_est + 0.10]
soluciones = []

for lam in lam_ejemplos:
    xi_sol, theta_sol, _, _, _ = resolver_bvp(lam, N)
    soluciones.append((lam, xi_sol, theta_sol))

# ============================================================
# 5. RECONSTRUCCIÓN APROXIMADA DE LA FORMA DE LA COLUMNA
#    dx/ds = sin(theta) ~ theta
#    dy/ds = cos(theta) ~ 1
#    Para una visualización simple usamos:
#    x_def = integral(theta dxi)
#    y_def = xi
# ============================================================
def reconstruir_forma(xi, theta, escala=0.12):
    x_def = np.zeros_like(xi)
    for i in range(1, len(xi)):
        x_def[i] = x_def[i-1] + 0.5*(theta[i] + theta[i-1])*(xi[i] - xi[i-1])
    y_def = xi.copy()
    return escala * x_def, y_def

# ============================================================
# 6. GRÁFICAS
# ============================================================
plt.figure(figsize=(9, 5))
plt.plot(lam_vals, max_theta, linewidth=2)
plt.axvline(lam_crit_est, linestyle='--', linewidth=1.5,
            label=fr"$\lambda_{{crit}} \approx {lam_crit_est:.3f}$")
plt.xlabel(r"$\lambda = \dfrac{wL^3}{EI}$", fontsize=12)
plt.ylabel(r"$\max |\theta(\xi)|$", fontsize=12)
plt.title("Amplificación de la respuesta del BVP")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(lam_vals, sigma_min_vals, linewidth=2)
plt.axvline(lam_crit_est, linestyle='--', linewidth=1.5,
            label=fr"$\lambda_{{crit}} \approx {lam_crit_est:.3f}$")
plt.xlabel(r"$\lambda = \dfrac{wL^3}{EI}$", fontsize=12)
plt.ylabel(r"Menor valor singular de $A(\lambda)$", fontsize=12)
plt.title("Casi singularidad de la matriz del sistema")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 6))
for lam, xi_sol, theta_sol in soluciones:
    plt.plot(theta_sol, xi_sol, linewidth=2, label=fr"$\lambda={lam:.3f}$")
plt.xlabel(r"$\theta(\xi)$", fontsize=12)
plt.ylabel(r"$\xi$", fontsize=12)
plt.title("Perfiles angulares para distintos valores de la carga propia")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(7, 8))
for lam, xi_sol, theta_sol in soluciones:
    x_def, y_def = reconstruir_forma(xi_sol, theta_sol, escala=0.15)
    plt.plot(x_def, y_def, linewidth=2, label=fr"$\lambda={lam:.3f}$")
plt.plot([0, 0], [0, 1], 'k--', linewidth=1, label="columna recta de referencia")
plt.xlabel("Desplazamiento lateral esquemático", fontsize=12)
plt.ylabel(r"$\xi = x/L$", fontsize=12)
plt.title("Formas aproximadas de la columna")
plt.grid(True, alpha=0.3)
plt.legend()
plt.axis("equal")
plt.show()

# ============================================================
# 7. COMPARACIÓN CON EL VALOR TEÓRICO CLÁSICO
# ============================================================
lam_greenhill = 7.8373
error_pct = abs(lam_crit_est - lam_greenhill) / lam_greenhill * 100.0

print(f"Valor clásico aproximado (Greenhill): lambda_cr ≈ {lam_greenhill:.4f}")
print(f"Estimación numérica obtenida aquí     : lambda_cr ≈ {lam_crit_est:.4f}")
print(f"Error relativo porcentual aproximado  : {error_pct:.2f}%")

# ============================================================
# 8. INTERPRETACIÓN FINAL
# ============================================================
print("\nInterpretación didáctica:")
print("- Este enfoque evita presentar el problema directamente como autovalores.")
print("- Se impone theta(0)=0 y theta'(1)=1, lo cual fija la escala.")
print("- Cerca de la carga crítica, el sistema lineal se vuelve casi singular.")
print("- Esa casi singularidad se observa en la caída de sigma_min y en el gran")
print("  crecimiento de la amplitud de la solución.")
print("- Esto permite introducir de forma intuitiva la idea de carga crítica.")


## 8. Comentarios finales para clase

Este enfoque tiene varias ventajas didácticas:

- El estudiante trabaja con un **problema de contorno estándar**.
- La discretización por diferencias finitas resulta directa.
- La condición $\theta'(1)=1$ elimina la indeterminación de escala.
- La cercanía al pandeo crítico se detecta observando la **amplificación** de la solución.
- Más adelante, cuando el grupo ya entienda bien este problema, se puede mostrar que la formulación homogénea natural conduce a un **problema de autovalores**.

## 9. Posibles extensiones

Como ejercicios posteriores, se podría pedir a los estudiantes:

1. Reemplazar la diferencia hacia atrás de primer orden por una de segundo orden en la punta.
2. Refinar la malla y estudiar convergencia de $\lambda_{crit}$.
3. Resolver el problema **no lineal** usando $\sin\theta$ en lugar de $\theta$.
4. Estudiar una columna con sección variable, donde $I(x)$ y $A(x)$ cambien con la posición.
5. Comparar el enfoque de barrido en $\lambda$ con una formulación explícita de autovalores.

